# Messages and Structured Output

**What you will build:** the skill of getting a model to return **typed, validated data** — a Python object you can trust — instead of free-form prose. This is the difference between a demo and something you can put in a program.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/02_messages_and_structured_output.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — same as notebook 0.1. Run this once per notebook.
!pip install -q openai pydantic

import os, getpass, json
from openai import OpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL = "meta-llama/llama-3.3-70b-instruct"   # change to any model on openrouter.ai/models
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])
print("Ready. Model:", MODEL)

## The problem with free text

In 0.1 the model returned a paragraph. That's fine to *read*, but impossible to *program against*. If you ask "classify this support ticket," you don't want three sentences — you want a field you can branch on:

```
free text:      "Well, this looks like it's probably a billing issue, though..."   ← unparseable
structured:     { "category": "billing", "urgency": "high" }                    ← usable
```

Getting that second form reliably is **structured output**, and it's the foundation the rest of the course stands on (tool arguments, agent state, evals — all structured). Let's see the naive way first, watch it break, then fix it.

## Attempt 1 — just ask for JSON (and see why it's not enough)

The obvious move: tell the model to reply with JSON.

In [ ]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Classify the support ticket. Reply with JSON: category and urgency."},
        {"role": "user",   "content": "I was charged twice for my subscription this month!"},
    ],
    temperature=0,
)
print(resp.choices[0].message.content)

You'll often get valid JSON — but sometimes it's wrapped in ```` ```json ```` fences, or prefaced with "Sure, here's the classification:", or a field is named `priority` instead of `urgency`. **You can't build on "often."** Two fixes stack on top of each other: force JSON at the API level, and validate the shape with Pydantic.

## Fix 1 — force JSON with `response_format`

Most providers support a JSON mode that guarantees the response is parseable JSON (no prose, no fences).

In [ ]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Classify the support ticket. Reply with JSON keys: category, urgency."},
        {"role": "user",   "content": "I was charged twice for my subscription this month!"},
    ],
    response_format={"type": "json_object"},   # ← guarantees valid JSON text
    temperature=0,
)
data = json.loads(resp.choices[0].message.content)   # now this always parses
print(type(data), data)

## Fix 2 — validate the *shape* with Pydantic

JSON mode guarantees *valid JSON*, not *the JSON you wanted*. The model might still omit `urgency` or return `urgency: "very high"` when you only allow three levels. [Pydantic](https://docs.pydantic.dev/) is the standard Python library for exactly this: declare the shape as a class, and it validates and coerces the data, raising a clear error when it doesn't fit.

```{note}
Pydantic is worth learning on its own — it's a dependency of a huge slice of the Python ecosystem, and the whole of **PydanticAI** (Block 1) is built on it. What you do by hand here, PydanticAI will do for you automatically.
```

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class TicketClassification(BaseModel):
    category: Literal["billing", "technical", "account", "other"]
    urgency:  Literal["low", "medium", "high"]
    summary:  str = Field(description="One-line summary of the issue")

# Validate the raw dict from Fix 1 against the schema:
ticket = TicketClassification.model_validate(data)
print(ticket)
print("category is now a real, checked field:", ticket.category)

## Putting it together: a reusable `extract` helper

A small function that takes any Pydantic model and returns a validated instance. We even feed the model the schema so it knows exactly which fields to produce. This pattern — *model + schema in, typed object out* — is the seed of everything in Block 1.

In [ ]:
def extract(model_class, user_text, instructions):
    """Call the LLM and return a validated instance of `model_class`."""
    schema = model_class.model_json_schema()
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system",
             "content": f"{instructions}\nReturn JSON matching this schema:\n{json.dumps(schema)}"},
            {"role": "user", "content": user_text},
        ],
        response_format={"type": "json_object"},
        temperature=0,
    )
    return model_class.model_validate_json(resp.choices[0].message.content)

result = extract(
    TicketClassification,
    "The app crashes every time I open the reports page. I need this fixed today.",
    "Classify the support ticket.",
)
print(result)

::::{dropdown} 🔧 Under the hood: what `response_format` changes
:color: info

`response_format={"type": "json_object"}` adds a constraint on the server side so the model's output is guaranteed to be syntactically valid JSON — no ```` ``` ```` fences, no "Sure, here's..." preamble. Some providers go further with `{"type": "json_schema", ...}`, which also enforces the *fields*. Support varies by model, which is exactly why we add the Pydantic validation layer on top: it makes the guarantee ours, not the provider's. That portability is the durable choice.
::::

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Add a field.** Give `TicketClassification` a `sentiment: Literal["angry", "neutral", "happy"]` and re-run `extract`. Notice you changed *one class* and the prompt updated itself (the schema is generated).
2. **Break it on purpose.** Change the model's instructions to omit a required field, and watch Pydantic raise a clear `ValidationError`. That error is your safety net.
3. **Extract something else.** Write a `Recipe` model (`title: str`, `ingredients: list[str]`, `minutes: int`) and extract it from a paragraph of cooking text.
::::

**What's next:** in **0.3** we give the model **tools** and write the agent loop by hand. Structured output is what makes it possible: a tool call is just structured output the model produces to ask for an action.